# HPFA — Historical Match Video Intelligence (Colab) V1

Two-stage video event pipeline with independent, reproducible validation.

| Stage | What it decodes | Signal | Purpose |
|---|---|---|---|
| A — triage | nothing (container packets) | inter-frame packet bytes / s | find dead-ball → restart patterns across the whole video in seconds |
| A — audit | — | seeded random windows | unbiased estimate of what triage misses |
| B — detail | only selected windows, 5 fps | visible pixel activity (camera-shift compensated) | event candidates with time bounds |

**Claim boundary.** Rows are `DERIVED_OBSERVATION` candidates. A row is *verified* only by an independent reference label (`VERIFIED_REFERENCE`) or a named human reviewer (`VERIFIED_HUMAN`). No tracking data → no tracking statistics. Capability is granted per event type only by held-out validation, and never transfers to a different footage domain. The "if played today" layer is a separate file and never edits observed rows.

**Rights.** Every video passes a rights manifest gate before decoding. This notebook contains no downloader for platform videos and must not be used with copies obtained by bypassing download/DRM/account protection.

Runtime: CPU is enough. A GPU is not used by the core pipeline.

## 0. Setup

In [ ]:
import os, subprocess, sys, json, time, pathlib
BRANCH = "ccr-09af526b-79apyv"   # change to main after merge
REPO = "https://github.com/Hikmetpinarbas/hpfa.git"
if not pathlib.Path("hpfa").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "hpfa"], check=True)
HPFA = pathlib.Path("hpfa").resolve()
SRC = HPFA / "hpfa/modules/core/historical_video_intelligence_lite/src"
sys.path.insert(0, str(SRC))
print(subprocess.run(["git", "-C", str(HPFA), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
%pip -q install numpy pytest huggingface_hub

## 1. Engineering self-test (synthetic fixture + HPFA integration)

Runs the module test-suite: contract, rights gate, two-stage mechanics on a generated video, reference evaluation, human review loop, counterfactual separation, and the **HPFA integration test** (`multiformat_file_inventory_lite` → `csv_surface_reader_lite` on the exported verified surface).

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    str(HPFA / "hpfa/modules/core/historical_video_intelligence_lite/tests")],
                   capture_output=True, text=True, cwd=HPFA)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0

## 2. Real-video validation data — SoccerTrack v2 (CC BY 4.0)

10 full university-level matches, panoramic fixed camera, human Ball Action Spotting labels (Shot, Goal, Free Kick, Throw In, Out, Player Successful Tackle, …).

Attribution: Scott, Uchida, Kuroda, Kim, Fujii — *SoccerTrack v2*, arXiv:2508.01802; dataset `atomscott/soccertrack-v2` on Hugging Face, CC BY 4.0. Changes: frames decoded and analysed; derived event candidates produced.

The Hugging Face repository is gated. **Request access with your own account on the dataset page and accept the terms**, then add a read token as Colab secret `HF_TOKEN` (🔑 icon in the left sidebar). This is the provider's intended access route.

Each half is ~3.3–3.6 GB. Default below: calibrate on one half, test on two different matches.

In [ ]:
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
DATA = pathlib.Path("/content/soccertrack_v2") if pathlib.Path("/content").exists() else pathlib.Path("soccertrack_v2")
CALIBRATE = ("117093", 1)
TESTS = [("118575", 1), ("128057", 2)]
SEGMENT_S = None   # e.g. 900 to analyse only the first 15 minutes of each half (faster)
HALF = {1: "1st", 2: "2nd"}
def fetch(match, half):
    for rel in (f"bas/{match}/{match}_12_class_events.json", f"videos/{match}/{match}_panorama_{HALF[half]}_half.mp4"):
        t0 = time.time()
        p = hf_hub_download("atomscott/soccertrack-v2", rel, repo_type="dataset", token=HF_TOKEN, local_dir=DATA)
        print(f"{rel}: {os.path.getsize(p)/1e9:.2f} GB in {time.time()-t0:.0f}s")
for m, h in [CALIBRATE] + TESTS:
    fetch(m, h)

## 3. Rights gate for the validation source

In [ ]:
from rights_gate import evaluate_rights, load_manifest
MANIFEST = HPFA / "configs/historical_video/validation/soccertrack_v2_rights_manifest.json"
print(json.dumps(evaluate_rights(load_manifest(MANIFEST)), indent=2))

## 4. Calibrate (in-sample) → test on held-out matches

Calibration searches Stage B thresholds on cached window series of the calibration half only. The reported numbers that count are the **held-out** ones.

In [ ]:
OUT = pathlib.Path("/content/hvi_validation" if pathlib.Path("/content").exists() else "hvi_validation")
cmd = [sys.executable, str(SRC / "validate_dataset.py"), "--data", str(DATA), "--manifest", str(MANIFEST),
       "--calibrate", f"{CALIBRATE[0]}:{CALIBRATE[1]}", "--out", str(OUT), "--domain", "PANORAMIC_FIXED_AMATEUR"]
for m, h in TESTS:
    cmd += ["--test", f"{m}:{h}"]
if SEGMENT_S:
    cmd += ["--segment-s", str(SEGMENT_S)]
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-4000:]); print(f"wall {time.time()-t0:.0f}s")

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUT / "validation_measurement_report_v1.md").read_text()))

## 5. HPFA integration on a real held-out surface

In [ ]:
for p in ("multiformat_file_inventory_lite/src", "csv_surface_reader_lite/src"):
    sys.path.insert(0, str(HPFA / "hpfa/modules/core" / p))
from multiformat_file_inventory import build_inventory
from csv_surface_reader import build_csv_surface_audit
m, h = TESTS[0]
surface = OUT / "heldout" / f"{m}_h{h}" / "hpfa_surface"
inv = build_inventory(surface)
audit = build_csv_surface_audit(surface, inv)
f = audit["files"][0]
print(audit["status"], audit["hard_block_hits"], f["field_bundle"])
for a in f["action_taxonomy"]:
    print(a["raw_type"], a["raw_subtype"], a["canonical_action_family_candidate"], a["surface_row_volume"])
assert audit["status"] != "FAIL_CLOSED" and not audit["hard_block_hits"]

## 6. Human review loop (for footage without reference labels)

`video_event_review_queue_v1.tsv` lists every candidate with a clip span. A reviewer fills `decision` (`CONFIRM` / `REJECT` / `RELABEL`), optional `team`/`player`, and **their name** in `reviewer`. Below: cut short review clips (local only, not published).

In [ ]:
import csv
run_dir = OUT / "heldout" / f"{m}_h{h}"
video = DATA / "videos" / m / f"{m}_panorama_{HALF[h]}_half.mp4"
clips = run_dir / "review_clips"; clips.mkdir(exist_ok=True)
with open(run_dir / "video_event_review_queue_v1.tsv") as fh:
    rows = list(csv.DictReader(fh, delimiter="\t"))
for row in rows[:5]:
    s, e = float(row["clip_start_s"]), float(row["clip_end_s"])
    subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y", "-ss", f"{s}", "-i", str(video), "-t", f"{e-s}",
                    "-vf", "scale=1280:-2", "-an", str(clips / f"{row['event_id']}_{row['event_type']}.mp4")], check=True)
print(sorted(p.name for p in clips.iterdir()))
# After filling the TSV:
# from hpfa_export import apply_review, write_event_outputs

## 7. Historical matches — source & rights review stage

Fenerbahçe–Benfica (25 Apr 2013) and Galatasaray 2000 (UEFA Cup final 17 May 2000; UEFA Super Cup 25 Aug 2000) have manifests in `configs/historical_video/rights_review/`. They are `REVIEW_REQUIRED` and the pipeline refuses them until the operator records a cleared rights basis (checklist R1–R6 inside each file). Reference events listed there are secondary-source candidates that need primary confirmation.

In [ ]:
for p in sorted((HPFA / "configs/historical_video/rights_review").glob("*.json")):
    v = evaluate_rights(json.loads(p.read_text()))
    print(p.name, "processing_allowed=", v["processing_allowed"], v["hard_block_hits"])

To process a historical copy **after** the review is cleared: put the lawful copy on your Drive, update the manifest (`rights_status`, `acquisition_method`, `download_protection_circumvented: false`, `attestation` or licence), then run with a broadcast domain label. Capability from the panoramic validation is capped at `EXPERIMENTAL` there; verification goes through the review queue (and, for goals, through the manifest's reference events after clock alignment).

In [ ]:
HIST_VIDEO = None       # e.g. "/content/drive/MyDrive/HPFA/archive/match.mp4"  (lawful copy only)
HIST_MANIFEST = HPFA / "configs/historical_video/rights_review/fenerbahce_benfica_2013_04_25.json"
if HIST_VIDEO:
    from pipeline import run
    cap = json.loads((OUT / "capability_heldout_v1.json").read_text())
    rep = run(HIST_VIDEO, json.loads(HIST_MANIFEST.read_text()), "/content/hvi_historical", capability=cap, domain="BROADCAST_SD_ARCHIVE")
    print(rep["status"], rep.get("hard_block_hits"))

## 8. "If played today" — separate counterfactual layer

Built only from verified observed rows, with explicit assumptions; numeric projections are refused without a registered, validated model.

In [ ]:
from counterfactual_layer import build_counterfactual, write_counterfactual
obs = json.loads((OUT / "heldout" / f"{m}_h{h}" / "video_events_v1.json").read_text())["event_rows"]
cf = build_counterfactual(obs, [{"id": "A1", "statement": "Modern restart routines shorten dead-ball time"},
                                {"id": "A2", "statement": "Modern pressing raises turnover rate after restarts"}],
                          "How would the restart rhythm differ if this match were played today?")
print(write_counterfactual(OUT / "counterfactual", cf)); print(cf["refusal_reason"], cf["observed_basis"]["verified_event_counts_by_type"])

## 9. Keep the evidence (your own Drive, not publication)

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT) + "_evidence", "zip", OUT)
print(archive)
# from google.colab import drive; drive.mount("/content/drive"); shutil.copy(archive, "/content/drive/MyDrive/HPFA/")